# Day 6-7 — Focal Loss vs Baseline

Testing three hypotheses from Day 5, one per failure-mode category found in the baseline:
1. Does focal loss's alpha term help iccad2's weak, near-threshold minority-class representation?
2. Does focal loss improve calibration enough to reduce the distribution-shift-driven false-positive floods on iccad3/4/5?
3. Does focal loss help iccad1 at all, given its failure looked like insufficient total data rather than an imbalance-specific problem?

Configuration used throughout: `gamma=2.0, alpha=0.75` (a first, unswept choice -- an alpha sweep is a planned follow-up, not yet run).

## iccad2 -- weak minority representation (Hypothesis 1)

| | Baseline | Focal loss (g2a75) |
|---|---|---|
| HS test mean | 0.474 | 0.653 |
| NHS test mean | 0.0105 | 0.0693 |
| Precision (full/clean) | 0.482 / 0.459 | 0.297 / 0.275 |
| Recall (full/clean) | 0.506 / 0.499 | 0.837 / 0.833 |

**Result: real, substantial improvement in recall, at a real cost to precision.** HS mean rose meaningfully (0.47 -> 0.65) -- the model became genuinely more confident about real hotspots, exactly what alpha=0.75 is built to do. Recall jumped from ~0.50 to ~0.84. But NHS mean also rose (0.01 -> 0.07), dragging precision down (0.48 -> 0.30) -- the model became more positive-biased overall, not selectively better at HS specifically.

**Verdict on Hypothesis 1**: partially confirmed, with an honest tradeoff. Given this project's earlier reasoning that recall matters more than precision for hotspot detection (a missed hotspot is far costlier than a false alarm), this tradeoff is plausibly desirable -- but it is a tradeoff, not a clean win, and should be reported as such rather than as unqualified improvement.

## iccad3 -- distribution shift (Hypothesis 2)

| | Baseline | Focal loss (g2a75) |
|---|---|---|
| HS test mean | 0.9657 | 0.9291 |
| NHS test mean | 0.2685 | 0.3559 |
| Precision (full/clean) | 0.128 / 0.125 | 0.112 / 0.103 |
| Recall (full/clean) | 0.970 / 0.965 | 0.978 / 0.973 |

**Result: focal loss did NOT fix the distribution-shift problem -- precision got slightly worse, not better.** NHS mean rose further (0.27 -> 0.36), amplifying the false-positive flood already present in the baseline. Recall improved marginally (already near-ceiling in the baseline).

**Verdict on Hypothesis 2**: not confirmed -- and with a clear, mechanistic reason why. Alpha's mechanism is a blanket bias toward flagging the positive class more; that is the wrong tool for a *calibration* problem (train/test base-rate mismatch), since it pushes the model to be even more positive-biased, which directly worsens a false-positive problem rather than correcting it. This is a genuinely informative negative result, not just "it didn't help" -- it points to *why* focal loss's specific mechanism is mismatched to this specific failure mode, and suggests calibration-focused techniques (e.g. post-hoc threshold recalibration accounting for the true test-time base rate, or temperature scaling) would be a more targeted fix than reweighting the loss function.

## iccad4, iccad5 -- pending

Both also fall into the distribution-shift category (Hypothesis 2) per Day 5 -- prediction, to be checked against the actual result once trained: given iccad3's outcome, expect a similar or worse precision trade for iccad4/5 too, since the underlying mechanism (alpha's positive-class bias worsening an already-miscalibrated model) is not benchmark-specific.

## iccad1 -- pending

Prediction: given the baseline's total collapse was attributed to insufficient total data (not addressable by loss-function reweighting), expect focal loss to show little to no improvement here -- worth running to confirm rather than skipping, since a confirmed negative result is still a useful, specific finding distinguishing "imbalance problem" from "data scarcity problem."

## Emerging conclusion

So far, focal loss (at gamma=2, alpha=0.75, unswept) helps the *weak representation* failure mode (iccad2) with a real recall/precision tradeoff, but does not help -- and may slightly worsen -- the *distribution shift* failure mode (iccad3). This is a meaningful, mechanistic finding: focal loss's alpha term addresses under-representation of a class during training, not miscalibration between train and test base rates -- these are different problems, and a single loss-function change does not address both. A lower alpha (less aggressive positive-class bias) is a planned follow-up to test whether it recovers some of iccad3's precision without giving up all of iccad2's recall gain.